# 02 — Preprocessing

Clean `Loan_default.csv`, encode features, apply SMOTE for class imbalance, and save processed output.

In [7]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [8]:
df = pd.read_csv("../data/raw/Loan_default.csv")

print("Dataset Shape :", df.shape)

df.head()

Dataset Shape : (255347, 18)


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [9]:
# Drop Index (ID column), remove duplicates
TARGET = "Defaulted?"

df = df.drop(columns=["Index"], errors="ignore")
df = df.drop_duplicates()
df = df.dropna(subset=[TARGET])

print("Cleaned Shape :", df.shape)
df.head()

KeyError: ['Defaulted?']

In [ ]:
X = df.drop(TARGET, axis=1)
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 3)
y shape: (10000,)


In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

print("Numerical Features:", num_cols)
print("Categorical Features:", cat_cols)

Numerical Features: ['Employed', 'Bank Balance', 'Annual Salary']
Categorical Features: []


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

Training Shape : (8000, 3)
Testing Shape  : (2000, 3)


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

transformers = [("num", numeric_pipeline, num_cols)]

if cat_cols:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    transformers.append(("cat", categorical_pipeline, cat_cols))

preprocessor = ColumnTransformer(transformers=transformers)

print("Preprocessor built")

Preprocessor built


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print("Train processed shape:", X_train_processed.shape)
print("Test processed shape :", X_test_processed.shape)

Train processed shape: (8000, 3)
Test processed shape : (2000, 3)


In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_processed,
    y_train
)

In [ ]:
print("Before SMOTE")
print(y_train.value_counts())

print()

print("After SMOTE")
print(pd.Series(y_train_smote).value_counts())

Before SMOTE
Defaulted?
0    7734
1     266
Name: count, dtype: int64

After SMOTE
Defaulted?
0    7734
1    7734
Name: count, dtype: int64


In [ ]:
joblib.dump(preprocessor, "../models/preprocessor.pkl")

print("Preprocessor Saved Successfully")

Preprocessor Saved Successfully


In [ ]:
processed_data = {
    "X_train":       X_train_processed,
    "X_test":        X_test_processed,
    "y_train":       y_train,
    "y_test":        y_test,
    "X_train_smote": X_train_smote,
    "y_train_smote": y_train_smote
}

joblib.dump(processed_data, "../models/processed_data.pkl")

print("Processed Data Saved")

Processed Data Saved


In [ ]:
print("=" * 50)
print("Preprocessing Completed Successfully")
print("=" * 50)
print("Training Samples :", X_train_processed.shape)
print("Testing Samples  :", X_test_processed.shape)
print("SMOTE Samples    :", X_train_smote.shape)

Preprocessing Completed Successfully
Training Samples : (8000, 3)
Testing Samples  : (2000, 3)
SMOTE Samples    : (15468, 3)
